In [6]:
# 1. setup
import os
import gzip
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kruskal, mannwhitneyu
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import warnings
warnings.filterwarnings('ignore')

base        = 'D:/TNBC_SV_DNA_Repair'
data_dir    = os.path.join(base, 'dataset')
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

group_order  = ['Low', 'Moderate', 'High']
group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}

print('setup done')
print('panel genes:', len(all_panel))

setup done
panel genes: 23


In [7]:
# 2. load clinical cohort
expr_clin = pd.read_csv(os.path.join(tables_dir, 'expr_tnbc_clinical.csv'), index_col=0)
cn_clin   = pd.read_csv(os.path.join(tables_dir, 'cn_tnbc_clinical.csv'),   index_col=0)
surv_clin = pd.read_csv(os.path.join(tables_dir, 'surv_tnbc_clinical.csv'))
mut_clin  = pd.read_csv(os.path.join(tables_dir, 'mut_tnbc_clinical.csv'))

clin_samples = list(expr_clin.columns)

print('samples:', len(clin_samples))
print('expression:', expr_clin.shape)
print('copy number:', cn_clin.shape)
print('survival:', surv_clin.shape)
print('OS events:', int(surv_clin['OS'].sum()))
print('PFI events:', int(surv_clin['PFI'].sum()))

samples: 112
expression: (26, 112)
copy number: (23, 112)
survival: (112, 11)
OS events: 18
PFI events: 18


In [8]:
# 3. expression disruption
expr_z    = expr_clin.apply(lambda row: (row - row.mean()) / (row.std() + 1e-8), axis=1)
expr_disr = (expr_z.abs() > 1.96).astype(float)
print('expression disruption shape:', expr_disr.shape)

expression disruption shape: (26, 112)


In [9]:
# 4. copy number disruption
cn_clin_panel = cn_clin.loc[all_panel, clin_samples]
cn_disr       = (cn_clin_panel != 0).astype(float)
print('copy number disruption shape:', cn_disr.shape)

copy number disruption shape: (23, 112)


In [10]:
# 5. mutation disruption
nonsynonymous = ['Frame_Shift_Del','Frame_Shift_Ins','Nonsense_Mutation',
                 'Splice_Site','Missense_Mutation']
mut_panel = mut_clin[mut_clin['gene'].isin(all_panel)]
mut_panel = mut_panel[mut_panel['effect'].isin(nonsynonymous)]

mut_disr = pd.DataFrame(0.0, index=all_panel, columns=clin_samples)
for _, row in mut_panel.iterrows():
    if row['gene'] in all_panel and row['sample'] in clin_samples:
        mut_disr.loc[row['gene'], row['sample']] = 1.0

print('mutation disruption shape:', mut_disr.shape)
print('mutated samples:', (mut_disr.sum(axis=0) > 0).sum())

mutation disruption shape: (23, 112)
mutated samples: 22


In [11]:
# 6. gene disruption score
gene_score_clin = (expr_disr + cn_disr + mut_disr) / 3.0
print('gene score shape:', gene_score_clin.shape)
print('mean disruption per gene:')
print(gene_score_clin.mean(axis=1).sort_values(ascending=False).head(10))

gene score shape: (26, 112)
mean disruption per gene:
HORMAD1    0.294643
RAD21      0.291667
RAD51B     0.279762
MLH3       0.264881
BRCA1      0.261905
RAD51      0.258929
BRIP1      0.252976
REC8       0.252976
BRCA2      0.247024
RAD51D     0.244048
dtype: float64


In [12]:
# 7. GIPS
gips_raw    = gene_score_clin.sum(axis=0)
gips_scaled = (gips_raw - gips_raw.min()) / (gips_raw.max() - gips_raw.min())

t33 = gips_scaled.quantile(0.333)
t66 = gips_scaled.quantile(0.667)

def assign_group(x):
    if x <= t33:   return 'Low'
    elif x <= t66: return 'Moderate'
    return 'High'

gips_clin_df = pd.DataFrame({
    'sample':     clin_samples,
    'GIPS_scaled': gips_scaled.values,
    'GIPS_group':  gips_scaled.apply(assign_group).values
})

print('GIPS groups:', gips_clin_df['GIPS_group'].value_counts().to_dict())
print('tertile boundaries: t33 =', round(t33, 3), ', t66 =', round(t66, 3))

gips_clin_df.to_csv(os.path.join(scores_dir, 'gips_scores_clinical.csv'), index=False)
gene_score_clin.to_csv(os.path.join(scores_dir, 'gene_disruption_scores_clinical.csv'))

GIPS groups: {'Moderate': 40, 'Low': 38, 'High': 34}
tertile boundaries: t33 = 0.483 , t66 = 0.655


In [13]:
# 8. disruption heatmap
fig, ax = plt.subplots(figsize=(14, 7))

order     = gips_clin_df.sort_values('GIPS_scaled')['sample'].tolist()
plot_data = gene_score_clin[order]

sns.heatmap(plot_data, cmap='YlOrRd', vmin=0, vmax=1,
            yticklabels=True, xticklabels=False,
            linewidths=0, ax=ax,
            cbar_kws={'label': 'disruption score'})

ax.set_title('Gene disruption scores — clinical TNBC cohort (n=112), sorted by GIPS')
ax.set_xlabel('patients (low to high GIPS)')
ax.set_ylabel('gene')
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb7_disruption_heatmap_clinical.png'), dpi=150)
plt.show()
print('saved heatmap')

saved heatmap


In [14]:
# 9. compare gene disruption scores original vs clinical cohort
gene_score_orig = pd.read_csv(os.path.join(scores_dir, 'gene_disruption_scores.csv'), index_col=0)

orig_mean  = gene_score_orig.mean(axis=1)
clin_mean  = gene_score_clin.mean(axis=1)
common     = sorted(set(orig_mean.index) & set(clin_mean.index))

r, p = spearmanr(orig_mean[common].values, clin_mean[common].values)
print('cross-cohort gene disruption concordance:')
print(f'Spearman r = {round(r, 3)}, p = {round(p, 4)}, n = {len(common)} genes')

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#d94f3d' if g in hr_genes else
          '#f0a500' if g in cohesin_genes else
          '#4878cf' for g in common]

ax.scatter(orig_mean[common].values, clin_mean[common].values,
           c=colors, s=70, alpha=0.85, edgecolors='white', linewidths=0.5)

for i, g in enumerate(common):
    ax.annotate(g, (orig_mean[g], clin_mean[g]),
                fontsize=7, alpha=0.75, xytext=(3, 3),
                textcoords='offset points')

z      = np.polyfit(orig_mean[common].values, clin_mean[common].values, 1)
x_line = np.linspace(orig_mean[common].min(), orig_mean[common].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'k--', alpha=0.4, linewidth=1)

ax.set_xlabel('mean disruption score — expression-threshold cohort (n=121)')
ax.set_ylabel('mean disruption score — clinical cohort (n=112)')
ax.set_title(f'Gene disruption concordance across cohort definitions\nSpearman r={round(r,3)}, p={round(p,4)}')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#d94f3d', label='HR'),
                   Patch(facecolor='#f0a500', label='Cohesin'),
                   Patch(facecolor='#4878cf', label='Meiosis')], fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb7_cohort_concordance.png'), dpi=150)
plt.show()
print(f'concordance: r={round(r,3)}, p={round(p,4)}')

cross-cohort gene disruption concordance:
Spearman r = 0.614, p = 0.0018, n = 23 genes
concordance: r=0.614, p=0.0018


In [15]:
# 10. survival analysis
surv_merged = gips_clin_df.merge(surv_clin, on='sample', how='inner')
print('survival merged shape:', surv_merged.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, tc, ec, label in [
    (axes[0], 'PFI.time', 'PFI', 'PFI'),
    (axes[1], 'OS.time',  'OS',  'OS'),
]:
    valid = surv_merged.dropna(subset=[tc, ec]).copy()
    valid[tc] = pd.to_numeric(valid[tc], errors='coerce')
    valid = valid.dropna(subset=[tc])

    kmf = KaplanMeierFitter()
    for grp in group_order:
        sub = valid[valid['GIPS_group'] == grp]
        if len(sub) < 3: continue
        kmf.fit(sub[tc], sub[ec], label=f'{grp} (n={len(sub)})')
        kmf.plot_survival_function(ax=ax, ci_show=True, color=group_colors[grp])

    if len(valid) > 5:
        res = multivariate_logrank_test(valid[tc], valid['GIPS_group'], valid[ec])
        lo  = valid[valid['GIPS_group'] == 'Low']
        hi  = valid[valid['GIPS_group'] == 'High']
        if len(lo) > 3 and len(hi) > 3:
            pw = logrank_test(lo[tc], hi[tc], lo[ec], hi[ec])
            ax.set_title(f'Clinical TNBC {label} (log-rank p={round(res.p_value,3)}, low vs high p={round(pw.p_value,3)})', fontsize=9)
        else:
            ax.set_title(f'Clinical TNBC {label} (log-rank p={round(res.p_value,3)})', fontsize=9)
    ax.set_xlabel('days')
    ax.set_ylabel('survival probability')
    ax.legend(fontsize=8)

plt.suptitle(f'Survival analysis — clinical TNBC cohort (n=112)', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb7_km_clinical.png'), dpi=150)
plt.show()
print('saved KM curves')

survival merged shape: (112, 13)
saved KM curves


In [16]:
# 11. Cox regression
cox_results = []
for tc, ec, label in [('PFI.time', 'PFI', 'PFI'), ('OS.time', 'OS', 'OS')]:
    valid = surv_merged[['GIPS_scaled', tc, ec]].dropna().copy()
    valid.columns = ['GIPS', 'duration', 'event']
    valid['duration'] = pd.to_numeric(valid['duration'], errors='coerce')
    valid = valid.dropna()
    if len(valid) < 10: continue
    try:
        cph = CoxPHFitter()
        cph.fit(valid, duration_col='duration', event_col='event')
        s     = cph.summary
        hr    = round(np.exp(s['coef'].values[0]), 3)
        ci_lo = round(np.exp(s['coef lower 95%'].values[0]), 3)
        ci_hi = round(np.exp(s['coef upper 95%'].values[0]), 3)
        p     = round(s['p'].values[0], 4)
        conc  = round(cph.concordance_index_, 3)
        print(f'{label}: HR={hr} (95% CI {ci_lo}-{ci_hi}), p={p}, concordance={conc}')
        cox_results.append({'endpoint': label, 'HR': hr,
                            'CI_low': ci_lo, 'CI_high': ci_hi,
                            'p': p, 'concordance': conc})
    except Exception as ex:
        print(f'{label}: {ex}')

cox_df = pd.DataFrame(cox_results)
cox_df.to_csv(os.path.join(tables_dir, 'clinical_cohort_cox.csv'), index=False)
print(cox_df)

PFI: HR=0.477 (95% CI 0.048-4.721), p=0.5271, concordance=0.509
OS: HR=0.393 (95% CI 0.042-3.702), p=0.4143, concordance=0.558
  endpoint     HR  CI_low  CI_high       p  concordance
0      PFI  0.477   0.048    4.721  0.5271        0.509
1       OS  0.393   0.042    3.702  0.4143        0.558


In [18]:
# 12. pathway enrichment
import gseapy as gp

high_mask = gips_clin_df[gips_clin_df['GIPS_group'] == 'High']['sample'].tolist()
low_mask  = gips_clin_df[gips_clin_df['GIPS_group'] == 'Low']['sample'].tolist()

high_mean = gene_score_clin[high_mask].mean(axis=1)
low_mean  = gene_score_clin[low_mask].mean(axis=1)
diff      = (high_mean - low_mean).sort_values(ascending=False)

print('top disrupted genes high vs low:')
print(diff.head(10))

enrichr_genes = diff[diff > 0].index.tolist()
print('genes submitted:', enrichr_genes)

enrich_results = {}
for lib in ['GO_Biological_Process_2023', 'KEGG_2021_Human']:
    try:
        res = gp.enrichr(gene_list=enrichr_genes,
                         gene_sets=lib,
                         outdir=None,
                         verbose=False)
        df  = res.results
        df  = df[df['Overlap'].apply(lambda x: int(x.split('/')[0])) >= 2]
        df  = df.sort_values('Adjusted P-value')
        enrich_results[lib] = df
        print(f'\n{lib} top 5:')
        print(df[['Term', 'Overlap', 'Adjusted P-value']].head(5).to_string())
        df.to_csv(os.path.join(tables_dir, f'enrichment_clinical_{lib}.csv'), index=False)
    except Exception as ex:
        print(f'{lib} failed: {ex}')

top disrupted genes high vs low:
HORMAD2    0.270898
BRIP1      0.260578
MLH3       0.252838
RAD51C     0.248710
BRCA1      0.233230
RAD51B     0.224458
CHEK2      0.222910
SMC1B      0.219298
SYCP3      0.216202
REC8       0.203818
dtype: float64
genes submitted: ['HORMAD2', 'BRIP1', 'MLH3', 'RAD51C', 'BRCA1', 'RAD51B', 'CHEK2', 'SMC1B', 'SYCP3', 'REC8', 'RAD51D', 'SYCP2', 'PALB2', 'ATM', 'SMC1A', 'BRCA2', 'STAG3', 'MSH5', 'RAD21', 'RAD51', 'STAG2', 'MSH4', 'HORMAD1']

GO_Biological_Process_2023 top 5:
                                                                   Term Overlap  Adjusted P-value
0                               Double-Strand Break Repair (GO:0006302)  11/168      2.834783e-15
1                                               DNA Repair (GO:0006281)  12/291      9.123666e-15
2                        Meiotic Sister Chromatid Cohesion (GO:0051177)    6/10      1.720263e-14
3  Double-Strand Break Repair Via Homologous Recombination (GO:0000724)   8/111      1.728419e-11
4

In [19]:
# 13. summary
orig_gips = pd.read_csv(os.path.join(scores_dir, 'gips_scores.csv'))

summary = {
    'clinical cohort size':           len(clin_samples),
    'OS events':                       int(surv_clin['OS'].sum()),
    'PFI events':                      int(surv_clin['PFI'].sum()),
    'GIPS groups':                     str(gips_clin_df['GIPS_group'].value_counts().to_dict()),
    'gene concordance r':              round(r, 3),
    'gene concordance p':              round(p, 4),
    'original cohort size':            len(orig_gips),
    'overlap with original':           len(set(clin_samples) & set(orig_gips['sample'])),
}

for k, v in summary.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(summary, orient='index', columns=['value']).to_csv(
    os.path.join(tables_dir, 'nb7_summary.csv'))
print('notebook 7 complete')

clinical cohort size: 112
OS events: 18
PFI events: 18
GIPS groups: {'Moderate': 40, 'Low': 38, 'High': 34}
gene concordance r: 0.614
gene concordance p: 0.4143
original cohort size: 121
overlap with original: 22
notebook 7 complete
